# Caption throughput trial: vLLM structured outputs (Step 5b item 3)

Captions the representative keyframe of every shot in the worker packages of
`VIDEO_IDS` with Qwen3-VL-4B-Instruct on vLLM, constrained to `CAPTION_SCHEMA`,
and writes `/kaggle/working/caption_vllm_report.json`.

Baseline (plan Progress, Step 5b item 2, transformers `generate` + xgrammar,
batch 16, one T4): 3.43 s/shot, valid JSON 1485/1511 (98.3%).
Same videos in the baseline: `L23_V010` 93 shots 231 s (2.48 s/shot),
`L23_V004` 107 shots 297 s (2.78 s/shot).

Prompt, schema, and `parse_caption` are imported from
`system1/src/system1/providers/captions.py` at `REPO_REF` (pure Python; the
package is not installed). Edit only the settings cell, then Run All.

In [ ]:
# Settings
REPO_URL = "https://github.com/awun0105/Multimodal-Agentic-Retrieval-Engine.git"
REPO_REF = "dev"
SOURCE_REPO = "1thesudden/AIOU26_release_test"  # repo holding the worker packages
RELEASE_ID = "canonical_release_v001"
BATCH_ID = "batch_000"
VIDEO_IDS = ["L23_V010", "L23_V004"]  # 93 + 107 shots
BATCH_SIZE = 16  # vLLM max_num_seqs; same as worker.yaml captions.batch_size
MAX_SHOTS = None  # for example 8 for a smoke run
MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"
REVISION = "ebb281ec70b05090aa6165b016eac8ec08e71b17"  # worker.yaml captions.revision
MAX_NEW_TOKENS = 700  # worker.yaml captions.max_new_tokens
MAX_IMAGE_SIDE = 512  # worker.yaml captions.max_image_side

In [ ]:
# Setup: secret, repository (import only), vLLM. The system1 package is not installed.
# vllm==0.11.0: first release line with Qwen3-VL support; wheels for torch 2.8 / CUDA 12.8.
# T4 (sm_75) caveats: no bf16 (dtype float16), no FlashAttention 2 (vLLM falls back to
# another attention backend), CUDA graphs can fail or eat memory (enforce_eager=True).
import os, subprocess, sys
from pathlib import Path

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

token = secret("HF_TOKEN")
if not token:
    raise RuntimeError("Add an HF_TOKEN secret with read access to the source repo")
os.environ["HF_TOKEN"] = token  # never printed

WORK = Path("/kaggle/temp" if Path("/kaggle").exists() else "/content")
OUT = Path("/kaggle/working" if Path("/kaggle").exists() else "/content")
REPO_DIR = WORK / "mare"
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--quiet", REPO_REF], cwd=REPO_DIR, check=True)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True).stdout.strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.11.0", "huggingface_hub",
                "pandas", "pyarrow", "pillow", "numpy"], check=True)
sys.path.insert(0, str(REPO_DIR / "system1" / "src"))
print("commit:", COMMIT)

In [ ]:
# Data: read shots, keyframes, and representative keyframe JPEGs from each worker ZIP in place
# (same selection rule as system1/src/system1/captions/video.py).
import io, zipfile
import pandas as pd
from PIL import Image
from huggingface_hub import hf_hub_download

def load_video(video_id):
    path = hf_hub_download(SOURCE_REPO, f"{RELEASE_ID}/worker/artifacts/{BATCH_ID}/{video_id}.zip",
                           repo_type="dataset", token=token, local_dir=WORK / "worker_zips")
    with zipfile.ZipFile(path) as archive:
        keyframes = pd.read_parquet(io.BytesIO(archive.read(f"{video_id}/keyframes.parquet")))
        shots = pd.read_parquet(io.BytesIO(archive.read(f"{video_id}/shots.parquet")))
        picks = keyframes[keyframes["is_representative"].astype(bool)].reset_index(drop=True)
        if set(picks["shot_id"]) != set(shots["shot_id"]):
            raise ValueError(f"{video_id}: some shots have no keyframe")
        if MAX_SHOTS:
            picks = picks.head(MAX_SHOTS)
        images = [Image.open(io.BytesIO(archive.read(f"{video_id}/{ref}"))).convert("RGB")
                  for ref in picks["keyframe_ref"]]
    return picks, images

videos = {video_id: load_video(video_id) for video_id in VIDEO_IDS}
for video_id, (picks, images) in videos.items():
    print(video_id, len(images), "shots")

In [ ]:
# Model. tensor_parallel_size=1: the 4B model in float16 (~9 GB) fits one 15 GB T4, the two
# T4s have no NVLink so tensor parallel adds PCIe all-reduce per layer, and the baseline used
# one model per T4. Production would run one engine per GPU, as the worker does.
import time
import vllm
from vllm import LLM, SamplingParams

load_started = time.monotonic()
llm = LLM(model=MODEL_ID, revision=REVISION, dtype="float16", tensor_parallel_size=1,
          max_model_len=4096, max_num_seqs=BATCH_SIZE, gpu_memory_utilization=0.90,
          enforce_eager=True, limit_mm_per_prompt={"image": 1, "video": 0}, seed=0)
load_seconds = round(time.monotonic() - load_started, 1)
print("vllm", vllm.__version__, "load", load_seconds, "s")

In [ ]:
# Run: one chat request per shot, greedy, constrained to CAPTION_SCHEMA.
import base64, json
from collections import Counter
from system1.providers.captions import CAPTION_PROMPT, CAPTION_SCHEMA, parse_caption

try:  # vLLM >= 0.11 name
    from vllm.sampling_params import StructuredOutputsParams
    sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS,
                              structured_outputs=StructuredOutputsParams(json=CAPTION_SCHEMA))
except ImportError:  # older name
    from vllm.sampling_params import GuidedDecodingParams
    sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS,
                              guided_decoding=GuidedDecodingParams(json=CAPTION_SCHEMA))

def message(image):
    image = image.copy()
    image.thumbnail((MAX_IMAGE_SIDE, MAX_IMAGE_SIDE))
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=95)
    url = "data:image/jpeg;base64," + base64.b64encode(buffer.getvalue()).decode()
    return [{"role": "user", "content": [{"type": "image_url", "image_url": {"url": url}},
                                         {"type": "text", "text": CAPTION_PROMPT}]}]

per_video, problems, samples = {}, Counter(), []
total_started = time.monotonic()
for video_id, (picks, images) in videos.items():
    started = time.monotonic()
    outputs = llm.chat([message(image) for image in images], sampling, use_tqdm=False)
    seconds = time.monotonic() - started
    results = [parse_caption(output.outputs[0].text) for output in outputs]
    valid = sum(result.payload is not None for result in results)
    for result in results:
        problems.update(result.problems)
    for pick, result, output in zip(picks.itertuples(), results, outputs):
        if len(samples) < 3:
            samples.append({"shot_id": pick.shot_id, "raw": output.outputs[0].text,
                            "problems": list(result.problems)})
    per_video[video_id] = {"shots": len(images), "valid": valid, "seconds": round(seconds, 1),
                           "seconds_per_shot": round(seconds / max(len(images), 1), 2)}
    print(video_id, per_video[video_id])
total_seconds = time.monotonic() - total_started

In [ ]:
# Report
shots = sum(row["shots"] for row in per_video.values())
valid = sum(row["valid"] for row in per_video.values())
report = {
    "commit": COMMIT, "vllm_version": vllm.__version__, "model_id": MODEL_ID, "revision": REVISION,
    "dtype": "float16", "tensor_parallel_size": 1, "enforce_eager": True, "max_model_len": 4096,
    "max_num_seqs": BATCH_SIZE, "max_new_tokens": MAX_NEW_TOKENS, "max_image_side": MAX_IMAGE_SIDE,
    "load_seconds": load_seconds, "videos": per_video, "shots": shots, "valid": valid,
    "valid_rate": round(valid / max(shots, 1), 4), "seconds": round(total_seconds, 1),
    "seconds_per_shot": round(total_seconds / max(shots, 1), 2), "problems": dict(problems),
    "first_captions": samples,
    "baseline_transformers_t4": {"seconds_per_shot": 3.43, "valid_rate": 0.983,
                                 "L23_V010": {"shots": 93, "seconds": 231},
                                 "L23_V004": {"shots": 107, "seconds": 297}},
}
(OUT / "caption_vllm_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps({key: value for key, value in report.items() if key != "first_captions"}, ensure_ascii=False, indent=1))